# Block 5 — Deep Learning with Time Series Data: Recurrent Neural Networks

**Goals for this block:**
- Build and train RNN model using Tensorflow
- Introduction to the Darts Library for time series forecasting
- Implement preprocessing steps from Block 1 using Darts
- Build and train RNN model using Tensorflow using Darts


## 0. Setup & Environment


In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=6500)])

import torch
assert torch.cuda.is_available()
torch.cuda.set_per_process_memory_fraction(0.3)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

import darts
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler


## 1. Data Loading & Preparation

We'll work with an **energy generation** dataset from Block 1. This synthetic dataset contains energy consumption patterns that exhibit both temporal dependencies and seasonality, making it ideal for time series forecasting tasks.

In [ ]:
# GUIDE: load and preprocess data

df = pd.read_parquet("energy_synthetic.parquet")
# Try to parse a timestamp column smartly
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.index.freq = 'h' 
df = df.set_index('timestamp').sort_index()

## 2. Training Recurrent Neural Networks with TensorFlow

In this section, we'll implement a Recurrent Neural Network (RNN) model using TensorFlow to forecast our time series data.

### 2.1 Data Preprocessing


In [ ]:
# 1. Handle missing values
filled = df.interpolate(method='linear')

# 2. Resample data
resampled = filled.resample("h").sum()

# 3. Split data into train and test sets
n_train = int(0.7 * len(resampled))
train = resampled.iloc[:n_train]
test = resampled.iloc[n_train:]

# 4. Scale data
scaler = StandardScaler()
train_scaled = pd.DataFrame(scaler.fit_transform(train), index=train.index, columns=train.columns)
test_scaled = pd.DataFrame(scaler.transform(test), index=test.index, columns=test.columns)

In [ ]:
# 5. Create sliding windows
def make_windows(series, window_size, forecast_horizon, stride):
    Xs, ys = [], []
    
    for i in range(0, len(series) - window_size - forecast_horizon + 1, stride):
        X_window = series[i:i + window_size]
        y_window = series[i + window_size:i + window_size + forecast_horizon]
        Xs.append(X_window)
        ys.append(y_window)
    
    return np.array(Xs), np.array(ys)


window_size = 24  # e.g., past 24 hours
forecast_horizon = 1  # e.g., next 1 hours
stride = 24  # e.g., move window by 24 hour


X_train, y_train = make_windows(train_scaled, window_size, forecast_horizon, stride)
X_test, y_test = make_windows(test_scaled, window_size, forecast_horizon, stride)
    


In [ ]:
# print shapes
print( "train_scaled shape:", train_scaled.shape)
print(f"X_train: (num_samples, window_size, num_features) -> {X_train.shape}\ny_train: (num_samples, forecast_horizon, num_features) -> {y_train.shape} ")

In [ ]:
# visualize a sample window and its forecast
sample_idx = 0
plt.figure(figsize=(12, 6))
plt.plot(range(window_size), X_train[sample_idx], label='Input Window')
plt.plot(range(window_size, window_size + forecast_horizon), y_train[sample_idx],
            label='Forecast Horizon', color='orange', marker='o')
plt.legend()
plt.title('Sample Sliding Window and Forecast Horizon')
plt.show()

### 2.2 Building the TensorFlow RNN Model

Now we'll create and train a simple RNN model using TensorFlow's Keras API.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

#### Model Architecture:
- Simple RNN with two stacked layers (50 units each)
- First RNN layer returns sequences to connect with the second layer (`return_sequences=True`)
- Output Dense layer with `forecast_horizon` units to generate predictions
- Adam optimizer with Mean Squared Error loss function

In [ ]:
model_tf = Sequential()
model_tf.add(SimpleRNN(units=50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model_tf.add(SimpleRNN(units=50, return_sequences=False))
model_tf.add(Dense(units=y_train.shape[2]))
model_tf.compile(optimizer='adam', loss='mean_squared_error')

**What’s happening**:

`SimpleRNN(units=50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2]))`

- SimpleRNN: a vanilla recurrent layer.
- units=50: the hidden_dim — 50 neurons → each time step’s hidden state has 50 features.
- input_shape=(timesteps, features): this tells Keras the shape of your input sequence:
- X_train_ex.shape[1] = number of time steps in each training sample
- X_train_ex.shape[2] = number of features per step
- return_sequences=True: means this layer will output the full sequence of hidden states, not just the last one.
    **Why?**
    Because you’re stacking another RNN layer next — it needs the entire sequence (not a single vector).

`Dense(units=y_train.shape[2])`
- A fully connected layer mapping from the RNN’s output vector → the desired target shape.
- units=y_train_ex.shape[1]: If you’re predicting a multivariate time series, this equals the number of output features per time step (e.g., 1 for univariate forecasting, >1 for multivariate).

### 2.3 Model Training

Let's train our TensorFlow RNN model with the preprocessed data.

In [ ]:
history = model_tf.fit(
    X_train, 
    y_train, 
    epochs=8, 
    batch_size=256, 
    verbose=1)


### 2.4 Making Predictions

Now we'll use our trained model to generate predictions on the test dataset.

In [ ]:
y_pred = model_tf.predict(X_test)

### 2.5 Model Evaluation

Let's evaluate our TensorFlow RNN model using appropriate metrics and visualization.

In [ ]:
from sklearn.metrics import mean_absolute_error 

# Inverse transform the predictions and actuals
y_pred_reshaped = scaler.inverse_transform(y_pred)
y_test_unscaled = scaler.inverse_transform(y_test.reshape(-1, y_test.shape[1]))

# Calculate MAE
print(f"Unscaled Mean Squared Error (MAE): {mean_absolute_error(y_test_unscaled, y_pred_reshaped)}")

# Plotting some predictions vs actuals
plt.figure(figsize=(12, 6))
plt.plot(y_test_unscaled[:100, -1], label='Actual', color='blue')
plt.plot(y_pred_reshaped[:100, -1], label='TF RNN Forecast', color='red')
plt.title('Actual vs Predicted Energy Consumption (first 100 Samples)')
plt.xlabel('Sample Index')
plt.ylabel('Energy Consumption')
plt.legend()
plt.show()


<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Train RNN with TensorFlow</h2>

Apply what you've learned on the `electricity dataset`:
1. preprocess the dataset
2. design your own RNN architecture
3. train a multivariate RNN model on the 100 costumers 
4. train a univariate RNN model on a specific customer (e.g "MT_001")
5. is the mean squarred error of the univariate model better than on the multivariate?

This exercise will help you design an RNN model and understand the training process.

</div>

In [ ]:
df_excercise = pd.read_parquet("electricity.parquet")
df_excercise['timestamp'] = pd.to_datetime(df_excercise['timestamp'], utc=True, errors='coerce')
df_excercise.index.freq = '15min' 
df_excercise = df_excercise.set_index('timestamp').sort_index()

# Uncomment the line below to select only the "MT_001" column for the exercise
# df_exercise = df_excercise[["MT_001"]]

In [ ]:
# Preprocess the exercise data: set the resampling freq to "hourly" and set the most appropriate aggregation method for the electricity data

# 1. Handle missing values


# 2. Resample data


# 3. Split data into train and test sets

# 4. Scale data

# 5. Create sliding windows
window_size = 24  # e.g., past 24 hours
forecast_horizon = 1  # e.g., next 1 hour
stride = 24  # e.g., move window by 24 hour
X_train_ex, y_train_ex = ...
X_test_ex, y_test_ex = ...

In [ ]:
# Build and train the TensorFlow RNN model on the exercise data

model_tf_ex = ...

In [ ]:
# Inverse transform the predictions and actuals and Calculate MAPE


# Plotting some predictions vs actuals


## 3. Training RNN Models with Darts

Now we'll explore the Darts library, which offers specialized time series functionality and simplifies the implementation of advanced forecasting models.

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Train RNN with DARTS</h2>

Explore the Darts library:
1. redesign the RNN model using DARTS lib
2. try out different hypermarameters   
   - Test different window sizes (e.g., 12, 24, 48)
   - Test different hidden layer dimensions (e.g. 100, 150)
   - Adjust the training parameters (e.g., epochs=10,20 ; batchsize=64, 128)

This exercise will help you optimize your model training process to achieve better forecasting results.
</div>

In [ ]:
# Preprocessing steps for the exercise dataset

# Convert to time series format for Darts
series = TimeSeries.from_dataframe(df_excercise)

# handle missing values
series = series.resample(freq = "h", method="sum")

# split into train and test
train_series, test_series = series.split_before(0.7)

# scale the data
scaler = Scaler(StandardScaler())
train_scaled = scaler.fit_transform(train_series)
test_scaled = scaler.transform(test_series)

### No Windowing needed for Darts RNN model

### 3.1 Building the Darts RNN Model

Darts provides a high-level API for time series forecasting models, including various RNN architectures.

In [ ]:
from darts.models import BlockRNNModel

#### Darts RNN Model Architecture:
- BlockRNNModel with vanilla RNN cells
- 2 stacked RNN layers with 50 hidden units each
- Adam optimizer with batch size of 256+

**Example Implementation:**

```python
from darts.models import BlockRNNModel

model_darts_ex = BlockRNNModel(
    model="RNN",                        # Use vanilla RNN cells
    input_chunk_length=window_size,     # 24 timesteps lookback
    output_chunk_length=forecast_horizon, # 1 timestep forecast
    hidden_dim=50,                      # 50 hidden units per layer
    n_rnn_layers=2,                     # 2 stacked RNN layers
    batch_size=256,                     # Batch size for training
    activation="adam",                  # Adam optimizer learning rate
    pl_trainer_kwargs={"accelerator": "gpu", "max_epochs": 50},
    random_state=42,
)
```
**What’s happening**:

- `model="RNN"`: Uses a vanilla Recurrent Neural Network cell (no gates like in LSTM or GRU).
- ``input_chunk_length: Number of past time steps the model uses as input. Defines the size of the “look-back window.
- `output_chunk_length`: Number of future time steps the model predicts at once. 

    💡 For "RNN" it is always set to one
- `hidden_dim`: Size of the hidden state vector i.e., how many features represent memory at each step.

    💡 Larger values → more capacity to learn complex temporal patterns.
- `n_rnn_layers`: Number of stacked RNN layers (depth of the recurrent network). The first layer extracts short-term patterns; the second learns higher-level temporal features.
- `batch_size`: Number of training sequences processed together before each gradient update.

    💡 Larger batches improve training speed but need more memory.
- `activation`: Activation function 

    💡 The Adam optimizer — an adaptive learning-rate method that works well for time series.

In [ ]:
# Build your own Darts RNN model

window_size = ...
forecast_horizon = 1

# model_darts_ex = BlockRNNModel(
#    ...


### 3.2 Model Training

Let's train our Darts RNN model on the preprocessed time series data.

#### Darts Library
You can easily train (`fit()`) and evaluate (`historical_forecast()`) on test data

**Example Implementation:**

```python
model.fit(
    series=train_data,
    epochs=num_epochs 
)

model.historical_forecasts(
    series=test_data, 
    forecast_horizon=forecast_horizon, 
    retrain=False, 
    last_points_only=True)
```

Think of `historical_forecasts()` as sliding-window prediction over your past data. Darts runs your model as if it were forecasting in real time, at multiple points in history.

It doesn’t just run `predict()` once. It moves along your `test_series` step by step, predicting ahead at multiple points.

`retrain`: This controls whether the model retrains at each step.

`last_points_only=True`: only return the last predicted point of each forecast window

In [ ]:
# Fit your model (think of increasing the epochs for better results)



### 3.3 Making Predictions

Now we'll generate forecasts using our trained Darts model.

In [ ]:
# Make predictions on the test set (use the historical_forecasts method)



In [ ]:
# Inverse transform the predictions and actuals
pred_series_unscaled = ...

### 3.4 Model Evaluation

Let's evaluate our Darts RNN model and compare its performance with the TensorFlow implementation.

In [ ]:
from darts.metrics import mae

# Calculate MAE
print("Darts RNN MAE:", mae(test_series, pred_series_unscaled))

start_idx = pd.Timestamp('2021-07-25 03:00:00')
end_idx = pd.Timestamp('2021-08-02 03:00:00')
# Plotting some predictions vs actuals
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Plot actual values on first subplot
test_series["MT_002"][-100:].plot(label="Actual", color='blue', ax=ax1)
pred_series_unscaled["MT_002"][-100:].plot(label="Darts RNN Forecast", color='red', title= "Actual vs Predicted Energy Consumption (last 100 Samples): Household MT_002", ax=ax1)

# Plot predictions on second subplot
test_series["MT_100"][-100:].plot(label="Actual", color='blue', ax=ax2)
pred_series_unscaled["MT_100"][-100:].plot(label="Darts RNN Forecast", color='red', title= "Actual vs Predicted Energy Consumption (last 100 Samples: Household MT_100", ax=ax2)

## ✅ Summary

## What You've Accomplished:
- Built and trained RNN models using TensorFlow/Keras
- Implemented preprocessing pipelines for both TensorFlow and Darts workflows
- Applied RNN techniques to energy consumption datasets
- Compared performance between TensorFlow and Darts implementations

These skills provide a foundation for building production-ready time series forecasting systems.